# 03 — Expert System Analysis

Implements and validates the rule-based expert system that encodes  
radiological knowledge about pneumonia diagnosis.

### Expert Rules:
1. **Opacity Rule** — elevated lung field intensity → consolidation
2. **Texture Rule** — GLCM features → parenchymal heterogeneity
3. **Density Distribution** — basal predominance → typical pneumonia pattern
4. **Consolidation Pattern** — focal high-density regions

### Objectives
1. Evaluate expert system standalone performance
2. Analyze which rules are most discriminative
3. Visualize lung segmentation and feature extraction
4. Compare expert scores between NORMAL and PNEUMONIA samples
5. Log results to MLflow

In [ ]:
import sys
sys.path.insert(0, '..')

import random
from pathlib import Path
from tqdm.notebook import tqdm

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import cv2
import mlflow

from src.models.expert_system import ChestExpertSystem, ExpertFindings
from src.utils.metrics import compute_clinical_metrics, print_clinical_report
from src.utils.visualization import plot_confusion_matrix, plot_roc_curve

plt.rcParams['figure.dpi'] = 120
DATA_ROOT = Path('../chest_xray')
mlflow.set_tracking_uri('../mlruns')

expert = ChestExpertSystem()
print('Expert system initialized')

## 1. Single Image Analysis Demo

In [ ]:
def load_rgb(path):
    img = cv2.imread(str(path))
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

random.seed(42)
normal_path    = random.choice(list((DATA_ROOT / 'test' / 'NORMAL').glob('*.jpeg')))
pneumonia_path = random.choice(list((DATA_ROOT / 'test' / 'PNEUMONIA').glob('*bacteria*')))

for path, true_label in [(normal_path, 'NORMAL'), (pneumonia_path, 'PNEUMONIA')]:
    img = load_rgb(path)
    findings = expert.analyze(img)
    
    print(f'\n{"="*60}')
    print(f'True Label: {true_label}')
    print(f'Prediction: {"PNEUMONIA" if findings.prediction == 1 else "NORMAL"}')
    print(f'Expert Score: {findings.final_score:.4f}')
    print(f'Confidence:   {findings.confidence:.4f}')
    print(f'\nSub-scores:')
    print(f'  Opacity:        {findings.opacity_score:.4f}')
    print(f'  Texture:        {findings.texture_score:.4f}')
    print(f'  Density:        {findings.density_score:.4f}')
    print(f'  Consolidation:  {findings.consolidation_score:.4f}')
    print(f'\nFindings:')
    for f in findings.findings:
        print(f'  • {f}')

## 2. Lung Segmentation Visualization

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
samples = [(normal_path, 'NORMAL'), (pneumonia_path, 'PNEUMONIA (Bacterial)')]

for row, (path, label) in enumerate(samples):
    img = load_rgb(path)
    gray = expert.preprocess(img)
    roi, mask = expert.extract_lung_roi(gray)
    
    axes[row,0].imshow(cv2.resize(img, (224,224)), cmap='gray')
    axes[row,0].set_title(f'Original\n{label}', fontweight='bold'); axes[row,0].axis('off')
    
    axes[row,1].imshow(cv2.resize(gray, (224,224)), cmap='gray')
    axes[row,1].set_title('CLAHE Preprocessed', fontweight='bold'); axes[row,1].axis('off')
    
    mask_resized = cv2.resize(mask.astype(np.uint8)*255, (224,224))
    axes[row,2].imshow(mask_resized, cmap='Blues')
    axes[row,2].set_title('Lung Mask', fontweight='bold'); axes[row,2].axis('off')
    
    roi_resized = cv2.resize(roi, (224,224))
    axes[row,3].imshow(roi_resized, cmap='gray')
    axes[row,3].set_title('Lung ROI', fontweight='bold'); axes[row,3].axis('off')

plt.suptitle('Lung Segmentation Pipeline', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/expert_lung_segmentation.png', bbox_inches='tight', dpi=150)
plt.show()

## 3. Expert System Evaluation on Test Set

In [ ]:
def evaluate_expert_on_split(split: str = 'test', max_per_class: int = 200):
    results = []
    for cls, label in [('NORMAL', 0), ('PNEUMONIA', 1)]:
        folder = DATA_ROOT / split / cls
        paths = list(folder.glob('*.jpeg')) + list(folder.glob('*.jpg'))
        sample = paths[:max_per_class]
        for path in tqdm(sample, desc=f'{split}/{cls}'):
            img = load_rgb(path)
            findings = expert.analyze(img)
            results.append({
                'true_label':        label,
                'prediction':        findings.prediction,
                'expert_score':      findings.final_score,
                'opacity_score':     findings.opacity_score,
                'texture_score':     findings.texture_score,
                'density_score':     findings.density_score,
                'consolidation_score': findings.consolidation_score,
                'confidence':        findings.confidence,
            })
    return pd.DataFrame(results)

print('Evaluating expert system on test set...')
df_expert = evaluate_expert_on_split('test')

metrics = compute_clinical_metrics(
    df_expert['true_label'].values,
    df_expert['prediction'].values,
    df_expert['expert_score'].values,
)
print_clinical_report(metrics, 'Expert System — Test Set Performance')

In [ ]:
# Confusion matrix
plot_confusion_matrix(
    df_expert['true_label'].values,
    df_expert['prediction'].values,
    title='Expert System Confusion Matrix',
    save_path='../reports/expert_confusion_matrix.png',
)

# ROC curve
plot_roc_curve(
    df_expert['true_label'].values,
    {'Expert System': df_expert['expert_score'].values},
    title='Expert System ROC Curve',
    save_path='../reports/expert_roc_curve.png',
)

## 4. Feature Importance Analysis

In [ ]:
score_cols = ['opacity_score', 'texture_score', 'density_score', 'consolidation_score']

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
palette = {0: '#66BB6A', 1: '#EF5350'}

for ax, col in zip(axes.flatten(), score_cols):
    for label, color in palette.items():
        data = df_expert[df_expert['true_label'] == label][col]
        cls_name = 'NORMAL' if label == 0 else 'PNEUMONIA'
        ax.hist(data, bins=30, alpha=0.6, label=cls_name, color=color, density=True)
    ax.set_title(col.replace('_', ' ').title(), fontweight='bold')
    ax.set_xlabel('Score'); ax.set_ylabel('Density')
    ax.legend()

plt.suptitle('Expert System Feature Distributions by Class', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('../reports/expert_feature_distributions.png', bbox_inches='tight', dpi=150)
plt.show()

In [ ]:
# Log expert system results to MLflow
mlflow.set_experiment('chest-xray-pneumonia')
with mlflow.start_run(run_name='expert_system_baseline'):
    mlflow.log_params({'system': 'rule_based', 'model': 'ChestExpertSystem'})
    mlflow.log_metrics({
        'test_accuracy':    metrics['accuracy'],
        'test_auc_roc':     metrics['auc_roc'],
        'test_sensitivity': metrics['sensitivity'],
        'test_specificity': metrics['specificity'],
        'test_f1':          metrics['f1_score'],
    })
    mlflow.log_artifact('../reports/expert_confusion_matrix.png')
    mlflow.log_artifact('../reports/expert_roc_curve.png')
    print('Results logged to MLflow')